In [1]:
# ############################################################
# Level 3 — 도매 고객 중 유별난 소비 고객 찾기 (공개 데이터)
# ############################################################
# ------------------------------------------------------------
# [목적] 도구 불러오기 — 표 / 전처리 / 모델
# ------------------------------------------------------------
import pandas as pd                               # 표(CSV) 다루기
from sklearn.preprocessing import StandardScaler  # 스케일링
from sklearn.ensemble import IsolationForest      # 이상치 탐지

In [2]:
# ------------------------------------------------------------
# [데이터 살펴보기 · EDA] 도매 고객 데이터 불러오고 살펴보기
#   · 고객 440명 × 품목별 연간 지출 (Fresh·Milk·Grocery 등)
# ------------------------------------------------------------
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv'
df = pd.read_csv(url)                              # 공개 URL에서 바로 (주소의 %20 = 빈칸)
print('데이터 크기(행, 열):', df.shape)           # (440, 8)
df.head()                                          # 앞 5줄 미리보기 (어떤 컬럼이 있나)

데이터 크기(행, 열): (440, 8)


,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,2,3,12669,9656,7561,214,2674,1338
1,2,3,7057,9810,9568,1762,3293,1776
2,2,3,6353,8808,7684,2405,3516,7844
3,1,3,13265,1196,4221,6404,507,1788
4,2,3,22615,5410,7198,3915,1777,5185


In [3]:
# ------------------------------------------------------------
# [데이터 살펴보기] 품목별 지출 요약 (값 범위가 커서 스케일링이 필요)
# ------------------------------------------------------------
feat = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']  # 분석할 품목 6개
df[feat].describe().round(0)                       # 평균·최대 등 (큰손 고객이 max를 크게 벌림)

,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
count,440.0,440.0,440.0,440.0,440.0,440.0
mean,12000.0,5796.0,7951.0,3072.0,2881.0,1525.0
std,12647.0,7380.0,9503.0,4855.0,4768.0,2820.0
min,3.0,55.0,3.0,25.0,3.0,3.0
25%,3128.0,1533.0,2153.0,742.0,257.0,408.0
50%,8504.0,3627.0,4756.0,1526.0,816.0,966.0
75%,16934.0,7190.0,10656.0,3554.0,3922.0,1820.0
max,112151.0,73498.0,92780.0,60869.0,40827.0,47943.0


In [4]:
# ------------------------------------------------------------
# [목적] 스케일링 -> 판정 -> 결과·점수를 표에 열로 붙이기
# ------------------------------------------------------------
X_s = StandardScaler().fit_transform(df[feat])    # 지출 단위가 커서 스케일링 (같은 잣대로)
iso = IsolationForest(contamination=0.05, random_state=0)
df['anomaly'] = iso.fit_predict(X_s)              # 판정: -1=이상, 1=정상
df['score']   = iso.decision_function(X_s)        # 점수: 낮을수록 더 이상
print('이상치 개수:', (df['anomaly'] == -1).sum(), '/', len(df))   # 개수 세기 (약 22명)

이상치 개수: 22 / 440


In [5]:
# ------------------------------------------------------------
# [목적] 이상 고객만 골라 '점수 낮은 순(가장 유별난)'으로 보기
# ------------------------------------------------------------
outliers = df[df['anomaly'] == -1]                # 이상(-1) 고객만 골라내기
outliers.sort_values('score')[feat + ['score']].head()   # 가장 유별난 고객부터 지출 확인

,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,score
183,36847,43950,20170,36534,239,47943,-0.234401
85,16117,46197,92780,1026,40827,2944,-0.217727
181,112151,29627,18148,16745,4948,8550,-0.207135
47,44466,54259,55571,7782,24171,6465,-0.204342
86,22925,73498,32114,987,20070,903,-0.159797


In [6]:
# ============================================================
# [결과 해석]
#  · 고객 440명 중 약 22명(5%)을 '유별난 소비' 고객으로 탐지
#  · score로 '가장 유별난' 고객부터 정렬해 우선 관리 대상을 고를 수 있음
#  · 특정 품목을 아주 많이(또는 적게) 쓴 도매 대형 거래처 등이 잡힘
#  · 활용: 사기 탐지, 불량 검출, 데이터 정제 등 (드물고 라벨 없는 것 찾기)
# ============================================================